# Item 99: Consider `memoryview` and `bytearray` for Zero-Copy

Interactions with `bytes`

## Notes

-   Python requires extra effort to parallelise CPU-bound computation
    (See [Item 79](../../Chapter_09/Item_079/item_079.qmd) and [Item
    94](../Item_094/item_094.qmd))
-   But, can support high-throughput parallel I/O (See [Item
    68](../../Chapter_09/Item_068/item_068.qmd) and [Item
    75](../../Chapter_09/Item_075/item_075.qmd))
-   However, understanding the tools available and how to use them
    *without* leading to slow code can require some skill
-   For example, consider a media-streaming server
    -   Users don’t need to download a video in advance
    -   Users can move forward or backward within a video
-   We might have functions to implement this by converting a time-code
    to a index and returning the associated chunk of data

In [1]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    # simulate by returning random data
    return os.urandom(size)


video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode)
size = (8**2)
video_data = request_chunk(video_id, byte_offset, size)

print(f"{video_id=}, {timecode=}, {byte_offset=}, {video_data=}")

video_id=1, timecode='01:09:14:28', byte_offset=0, video_data=b'v\x86\x84\x8bbk\x9b\x95AI\xc1]9\x94G\xc4\xef\xc9z\xdbJYDU\xbas\xfa\xcc1#\xc7J\x03\x13\x15N\xd2\x1fI"\x14DI\x99\x03Fl\x87\x90y*\xb6h\xca\x87\x1f\xe7\xa6y\xc1\xaa\x17q\xb6'

-   How do we now implement the server-side handler that receives
    `request_chunk`
    -   Must then return the associated video data chunk
-   First we assume that the program is driven by an `asyncio` process
    (See [Item 76](../../Chapter_09/Item_076/item_076.qmd))
    -   Now want to focus on how to handle extracting the chunk
    -   Assume video is cached memory
    -   Extracted then sent over a socket back to a client

In [2]:
import os # for demo only

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder


def request_chunk(video_id, byte_offset, size):
    # Returns size bytes of video_id's data from the offset
    return video_data[byte_offset : byte_offset + size]

# Adding in the handling

# simulate a socket connection
class NullSocket:
    def __init__(self):
        self.handle = open(os.devnull, "wb")

    def send(self, data):
        self.handle.write(data)

socket = NullSocket() # represents client socket connection
size = (8 ** 2) # Requested chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video_id

video_id = 1
timecode = "01:09:14:28"

byte_offset = timecode_to_index(video_id, timecode)
chunk = request_chunk(video_id, byte_offset, size)
socket.send(chunk)

print(f"Sent {chunk=} over socket")

Sent chunk=b'\x91a\x1b2\xfc\x16\xf56\xfem\xab\t\xda\x8c\xfd\x10\n\xb3\xc7^\x8f\x9d>\x7f\xcd\x11a\x89>\xdc\x8c\x86(\xf4\x86\xa9-\xd50DN1\x8e\x96e\r\x15\xbd\xea\xf8\xb8\xd4\xeb\xc9\x04S\x0c\xff"v3\x80v\xf5' over socket

-   Latency and throughput determined by two factors
    1.  How long to slice the chunk from `video_data`
    2.  How long to transmit over a socket
-   Focusing just on point 1, we can microbenchmark how long fetching a
    chunk takes.
    -   We’ll also exclude the function call wrapper
    -   Here we’ll set the size to $20$ MB.

In [3]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
byte_offset = 0

def run_test():
    chunk = video_data[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000979977 seconds

-   This takes about $5$ milliseconds
-   Theoretical server maximum throughput is thus, limited by video
    extraction speed as

$$
\begin{align}
    \frac{20 \text{ MB}}{5 \text{ ms}} &= 4 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Server also limited to,

$$
\begin{align}
    \frac{1 \text{ CPU=second}}{5 \text{ ms}} &= 200 \text{ clients in parallel}
\end{align}
$$

-   But we already know that `asyncio` should be able to scale up to
    tens of thousands of simultaneous connections
-   The slowdown is because as discussed slices create copies
    -   Copying consumes CPU time
-   Instead we can use `memoryview`
    -   A built-in type for handling the CPython `buffer` protocol
        -   Low-level C API allowing Python runtime and C extensions
            (See [Item 96](../Item_096/item_096.qmd)) to access
            underlying data buffers
            -   Can then treat them as `bytes` instances
        -   Since Python 3.12 the buffer protocol is also emulatable in
            python
-   `memoryview` can be sliced to create a new `memoryview` without a
    copy

In [4]:
data = b"shave and a haircut, two bits"
view = memoryview(data)
chunk = view[12:19]

print(chunk)
print("Size:            ", chunk.nbytes)
print("Data in view:    ", chunk.tobytes())
print("Underlying data: ", chunk.obj)

Size:             7
Data in view:     b'haircut'
Underlying data:  b'shave and a haircut, two bits'

-   These *zero-copy* operations can significantly speed-up code that
    heavily processes memory, e.g.
    1.  I/O-bound access
    2.  Heavy numerical mathematics (e.g. Numpy)
-   Using `memoryview` as a drop-in replacement for our video serving
    service

In [5]:
import timeit

size = 20 * (1024**2)
video_data = os.urandom(20 * size) # Bytes containing data for
video_view = memoryview(video_data)
byte_offset = 0

def run_test():
    chunk = video_view[byte_offset : byte_offset + size]

result = ( timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100 )

print(f"{result:0.9f} seconds")

0.000000200 seconds

-   This should run in a several hundred nanoseconds
-   So an order of magnitude faster than the `bytes` slicing technique
-   Our new theoretical maximum throughput is then

$$
\begin{align}
\frac{20 \text{ MB}}{250 \text{ ns}} &= 80 \text{ TB}\text{s}^{-1}
\end{align}
$$

-   Or in terms of parallel clients

$$
\begin{align}
\frac{1 \text{ CPU-second}}{250 \text{ ns}} &= 4 \times 10^{9}
\end{align}
$$

-   So four million clients. Now the program should be bound by the
    socket performance rather than CPU constraints.

-   Now consider a reversed process

    -   Users must submit live video streams that are then broadcast out
        to viewers

-   We need to store incoming video data

    -   Cache it for clients to read from

In [6]:
import os

def timecode_to_index(video_id, timecode):
    # Returns byte offser in the video data
    return 0  # placeholder

# socket connection from client


size = (4 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]

video_id = 1
timecode = "01:09:14:28"
byte_offset = timecode_to_index(video_id, timecode) # Incoming buffer position
video_view = memoryview(video_cache)


class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
chunk = socket.recv(size)
before = video_view[:byte_offset]
after = video_view[byte_offset + size:]

new_cache = b"".join([before, chunk, after])

print(f"Updated the cache: {new_cache=}")

Updated the cache: new_cache=b"^\xff(\xe05>7\xe6\x95\x05\xf9\xb1$\x96\x14\xf3\xeb\x81\xa0\xa4\xe9\xb7\xf1\x0b\x88\x9e\x14\xe38N{\x12\xdd\xcf\x0e\x1a\x1d\x9e\x1c\x84\xe8\x8c\xba\x80\xe2\xde\xad\n\xa3\x88\xcb\xa0\xd4\xa8\x03\x81\x7f\x1a\xe1\x1e\xbd\xf4\xbc\xa2\xeb\xb9{\xe3'\x12\xac<\x97\xfe\xb1\x87\xc4\xb4\xd1y\x1a\xd4qj\xfe\xcd\x927\x94h\xbc,\x98\xba\x17\xc9\x98u\xa8\xea\xb7\xc7\xd4c`p\x18\x84\xee\xf1\xd0\x86>\x10h.\xc5W}U\x959a\x10\x1a*\xcc\xf4k_y0\xd4\x81n(\x8fF\xd8O\x86\x0fT\x9d%}\xc3/\x9b\x10\xde\xf0\xef~\xb3ZN*\r\xab\xdb\xe2\xa0$\xa4\xd0\x1b\xf2\xe9\x8b0\xf1\x06\xa9\x80\xbb\xf4@BH\xa0!\xe3\xc4\xa4Z\xcf*\x18\xee]\xa9\xdb\xd8~PV&\x80u-\x9b\x81\xbd\x80wS\xbb2w\xff\x88\xb9B\xf5\xf6b\xb6\xf9\xa4\x00\xd7p\xe0\xdf\x9d\xfck\xc5\x1e\xa31\xf4\x16\xa7F\xe8\xf56\x86\x98Tn/\x85\xe9\xa0/H.?\x83\xb1\x17'G\x9aVf\xfa\xb2\xa9P\xfa)\xbcfl\x00/\x97\x1e\x85\x05\x1dOAq\xc5\x84\xd7]\xd6\x01A`d\x020\xd7\x8e\x88\xc6\xc2\x8cs\x87\xf4\x0f\xfc\xc9s\xf9\x10x\x16O_\xa3z\xcf\xe7-+\xad\xd4\x8f\t\xb6{"

-   `socket.recv` returns a `bytes` instance
    -   Splice this into the existing cache
    -   Insert at the current `byte_offset` via slicing and `bytes.join`
-   Now need to profile the timing

In [7]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

socket = MockIncomingSocket()
size = (1024 ** 2) # Incoming chunk size
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)
byte_offset = 1234 # pick arbitrary point in the middle

def run_test():
    chunk = socket.recv(size)
    before = video_view[:byte_offset]
    after = video_view[byte_offset + size : ]
    new_cache = b"".join([before, chunk, after])

result = (timeit.timeit(stmt="run_test()", globals=globals(), number=100,) / 100)

print(f"{result:0.9f} seconds")

0.000956511 seconds

-   This takes about three milliseconds to receive $1$ MB and update the
    cache.
-   Maximum throughput to receive is then

$$
\begin{align}
\frac{1 \text{ MB}}{ 3 \text{ ms}} &\approx 330 \text{ MB}\text{s}^{-1}
\end{align}
$$

-   Means we are limited to about $300$ simultaneously streaming clients
-   Can use `bytearray` instead of `memoryview`
    -   `bytes` are immutable like strings

In [8]:
some_bytes = b"hello"
some_bytes[0] = 0x79

-   `bytearray` is effectively a mutable version of `bytes`
    -   Can overwrite indices
-   `bytearray` values are integers rather than bytes

In [9]:
array = bytearray(b"hello")
array[0] = 0x79
print(array)

bytearray(b'yello')

-   Can still wrap a `bytearray` in a `memoryview` to avoid extra copies
    -   Then can slice the `memoryview` and modify to overwrite the
        underlying `bytearray`

In [10]:
array = bytearray(b"row, row, row your boat")
view = memoryview(array)
write_view = view[3:13]
write_view[:] = b"-10 bytes-"
print(array)

bytearray(b'row-10 bytes- your boat')

-   Library methods in Python user the buffer protocol for fast data
    receipt or reading, e.g.
    1.  `socket.recv_into`
    2.  `RawIOBase.read_into`
-   These methods avoid creating copies and allocating memory
    -   Received data goes into existing buffer
-   We can convert our program to use `recv_into` and a `memoryview`
    slice to speed up our broadcasting method

In [11]:
class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (4 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)

chunk = write_view[byte_offset : byte_offset + size]
socket.recv_into(chunk)

print(f"New cache: {video_cache=}")

New cache: video_cache=b'\xc2=\xfe\x99\xbcQ\xd4\x16D\xc4\xeeS\xf1\xee@\xd6\x93z(\xcf\xc1\xeeg\xc3\\9}\xcd2Bf\xc0\xeeo^\xa3\x86\xe0L\xee1$\xd3\xcc\xb9z\xa1u\x1dW\x11\xa2$\x1ejc\xf6\x12f~\xdf\xa2+\xa6\xb1\xe3\x92\x81D\xdbl\x1e\xee\x1d\xcc>\xd8\x80\xa4\xb1;\xc7\xb3\xbd\xe9\xa7\xe1#86c2q\xc6-\xad\xd401o&!\xbc\x0e\x82}_6\x91br\xd3\xa6<\xc7\xeey\x9a_m\x89\xddX\xb6\t l\xdd\xce\x91\x82\xecb\xaf\xea\x14\xe2hu~\xe7\x11\xbf\x91Y2\xf3\x97q\x1a"m/\xd3\x91\xd7^\xa0Z\x9b\xa2\xb2\x94\xee\x86\xedj\xacK\xfc\xa8\xc0w\xfb\x86\x1d^\xc5\x02\xa4-`\xef\x7f\xf7f\x96\xfa<\xc6\x85\x88Ym(\xec\xe5\xb5*\xc0\xf6\x82\x11\xa9\xdf\xc0I\x96\xe9\x0fd\xe7{\xca\xca8\xb4\xf3l\xa7lpC\xd0\xd8\x1ad\xb3\x0c\xe1E\x92*\xa7}\x08\xe1`\xcb\xbeD\xb92v@"TZ~\x05\x97\x98\x89\xe2\xf00\xe41|\xc3\xf3\xed\x03\xdb\xc0E\xb9\'TP\xbb7\xdc\x06\xfa\xd9\xc5\xd5\xef\xab\xe1\xa1\xdc\xb3[`}c \xf1S2\xe5\x10\xa9\xa60aAdw\x9c\x88x\xb2\xa1Fut\xba\xed\x02\xb3\xf6nL\x96\x189G'

-   We can again microbenchmark the result for a $1$ MB chunk

In [12]:
import timeit
import os

class MockIncomingSocket:

    def recv(self, size):
        return video_view[byte_offset : byte_offset + size]

    def recv_into(self, buffer):
        source_data = video_view[byte_offset : byte_offset + size]
        buffer[:] = source_data

# socket connection from client
socket = MockIncomingSocket()

size = (1024 ** 2) # Incoming chunk size
byte_offset = 1234
video_data = os.urandom(20 * size) # Bytes containing data for video
video_cache = video_data[:]
video_view = memoryview(video_cache)

video_array = bytearray(video_cache)
write_view = memoryview(video_array)


def run_test():
    chunk = write_view[byte_offset : byte_offset + size]
    socket.recv_into(chunk)

result = (
    timeit.timeit(stmt="run_test()", globals=globals(), number=100) / 100
)

print(f"{result:0.9f} seconds")

0.000033138 seconds

-   On my machine this takes about $90 \;\mu\text{s}$. Which means we
    could support,

$$
\begin{align}
    \frac{1 \text{ MB}}{90 \; \mu\text{s}} &= 11 \text{ GB}\text{s}^{-1}
\end{align}
$$

-   Which also supports,

$$
\begin{align}
    \frac{11 \text{ GB}}{1 \text{MB}} &= 11,000 \text{ processes}
\end{align}
$$

-   Much better scalability

## Things to Remember

-   `memoryview` provides zero-copy methods for reading and writing to
    slices of objects supporting the buffer protocol
-   `bytearray` built-in provides a mutable `bytes`-like type
    -   Can be used for zero-copy data reads
    -   Works with functions like `socket.recv_into`
-   `memoryview` can wrap a `bytearray`
    -   Let’s received data to be spliced into an existing buffer
    -   No need for extra copies